In [7]:
import numpy as np

def calc_mse(y):
    '''calcualte mean squared error'''
    if len(y)==0:
        return 0.0

    mean_value=np.mean(y)
    return np.mean((y-mean_value)**2)

def find_best_split(x,y,min_samples_leaf=1):
    '''search all numerical feature thresholds'''
    n_sample,n_features=x.shape
    parent_mse=calc_mse(y)
    best_gain=0.0
    best_split=None

    for index in range(n_features):
        values=np.unique(x[:,index])
        if len(values)<2:
            continue

        thresholds=(values[:-1]+values[1:])/2.0

        for threshold in thresholds:
            left=x[:,index]<=threshold
            right=~left
            n_left=left.sum()
            n_right=right.sum()
            if (n_left<min_samples_leaf or n_right<min_samples_leaf):
                continue
            left_mse=calc_mse(y[left])
            right_mse=calc_mse(y[right])
            weighted_mse=(n_left/n_sample)*left_mse+(n_right/n_sample)*right_mse
            gain=parent_mse-weighted_mse
            if gain>best_gain:
                best_gain=gain
                best_split=(index,threshold)

    return best_split,best_gain

In [8]:

X_demo = np.array([
    [1], [2], [3], [4], [5], [6]
], dtype=float)

y_demo = np.array([
    100, 110, 120, 300, 310, 320
], dtype=float)

print("Parent MSE:", calc_mse(y_demo))

split, gain = find_best_split(X_demo, y_demo)

print("Best split:", split)
print("MSE reduction:", gain)


Parent MSE: 10066.666666666666
Best split: (0, np.float64(3.5))
MSE reduction: 10000.0


In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.datasets import fetch_california_housing

housing=fetch_california_housing()
x=housing.data
y=housing.target

print(x.shape)
print(y.shape)

(20640, 8)
(20640,)


In [11]:
x=pd.DataFrame(x,columns=housing.feature_names)
y=pd.DataFrame(y,columns=['target'])

In [12]:
x.describe()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
count,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,3.870671,28.639486,5.429000,1.096675,1425.476744,3.070655,35.631861,-119.569704
std,1.899822,12.585558,2.474173,0.473911,1132.462122,10.386050,2.135952,2.003532
min,0.499900,1.000000,0.846154,0.333333,3.000000,0.692308,32.540000,-124.350000
25%,2.563400,18.000000,4.440716,1.006079,787.000000,2.429741,33.930000,-121.800000
50%,3.534800,29.000000,5.229129,1.048780,1166.000000,2.818116,34.260000,-118.490000
75%,4.743250,37.000000,6.052381,1.099526,1725.000000,3.282261,37.710000,-118.010000
max,15.000100,52.000000,141.909091,34.066667,35682.000000,1243.333333,41.950000,-114.310000


In [13]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
len(x_train)

16512

In [22]:
model=DecisionTreeRegressor(criterion='squared_error',min_samples_leaf=2,max_leaf_nodes=3,max_features='log2')

In [23]:
model.fit(x_train,y_train)
y_pred=model.predict(x_test)

from sklearn.metrics import r2_score
print('r2_score:',r2_score(y_test,y_pred))

r2_score: 0.33187923229517036
